# 18.10 蒸餾是否幫到學生？


蒸餾學生答對更多題，是否一定是教師訊號的功勞？如果它比基準多讀了一倍資料，或學生架構也變大，答案還不能歸因蒸餾。前置是[學生大小](https://birdhackor.github.io/tiny-perceptron-vlm/18.2.html)和[真值與教師混合](https://birdhackor.github.io/tiny-perceptron-vlm/18.9.html)。比較應讓同一個學生架構、初始化、訓練資料與已讀token預算分成兩條路：普通SFT與加入蒸餾，最後在同一份獨立真值上評估。SFT是監督式微調，以理想助手回答教學生；加入蒸餾則另外讀教師的候選比例。這份獨立評估題没有用來更新學生，答案真值也不由教師的猜測代填。

教師計算並不免費。假設兩種方法學生部分各花10秒，訓練100步、讀1000個token；蒸餾另花40秒教師forward（前向計算：把輸入交給教師，算出候選分數與比例），總時間就是50秒，不是10秒。按學生更新預算比較能回答「多給教師訊號有沒有幫忙」；按同樣總時間比較則回答另一個問題，蒸餾可能只能做較少更新。兩種公平應分開報。

下面用人工統計記錄展示如何讀表，不是執行模型訓練。把正確數與測試數放入記錄，算出正確率；同時把學生與教師時間加起來，避免把教師成本藏起來。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
runs = [
    {
        "method": "SFT",
        "student_params": 6104,
        "steps": 100,
        "tokens": 1000,
        "student_seconds": 10,
        "teacher_seconds": 0,
        "correct": 3,
        "total": 4,
    },
    {
        "method": "distill",
        "student_params": 6104,
        "steps": 100,
        "tokens": 1000,
        "student_seconds": 10,
        "teacher_seconds": 40,
        "correct": 4,
        "total": 4,
    },
]
for run in runs:
    accuracy = run["correct"] / run["total"]
    seconds = run["student_seconds"] + run["teacher_seconds"]
    print(
        run["method"],
        "參數",
        run["student_params"],
        "步數",
        run["steps"],
        "token",
        run["tokens"],
        "正確率",
        accuracy,
        "總秒數",
        seconds,
    )

輸入字典已經有實驗條件與結果，程式只計算衍生指標。輸出SFT是6104參數、100步、1000token、0.75、10秒；蒸餾同樣學生預算、1.0、50秒。這份構造數字只能說明報告應呈現什麼，不能當作蒸餾實際提升證據；要證實改善，必須訓練並保存真實結果，再用足夠獨立題目與重複實驗檢查。

評價不能只用「與教師回答一致率」。教師若給錯答案，學生模仿得更像可能反而退步；應用真值判內容，用格式規格判結構，並重跑既有誠實與安全情境。同樣正確率下，也可能一組善於短題、另一組在長題失敗，所以應按題型或輸入條件拆開看。

若教師先生成離線文字，成本改成資料生成與驗證，蒸餾訓練時不一定每步教師forward；仍要另外列出這些一次性成本。正式白盒訓練則必須有可載入、已驗證能力的教師權重，不能把本章的隨機教師檢查當作可用教師。需要保存版本與[詞表對齊](https://birdhackor.github.io/tiny-perceptron-vlm/18.6.html)條件，讓比較能重現。

練習只把蒸餾的teacher_seconds改成0。先預測正確率不變、總時間變10，執行核對後解釋：你只改了記帳假設，沒有訓練新學生。如果真的預先快取教師訊號，還應補報生成快取所花時間與儲存，不可把成本抹去。
